In [1]:
import numpy as np


class DecisionTreeNode:

  def __init__(
      self,
      feature=None,
      threshold=None,
      left=None,
      right=None,
      *,
      value=None,
  ):
    self.feature = feature
    self.threshold = threshold
    self.left = left
    self.right = right
    self.value = value

  def is_leaf_node(self):
    return self.value is not None


class DecisionTreeScratch:

  def __init__(self, max_depth=10, min_samples_split=2, max_features=None):
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.max_features = max_features
    self.root = None

  def _gini(self, y):
    if len(y) == 0:
      return 0
    p = np.bincount(y) / len(y)
    return 1.0 - np.sum(p**2)

  def _best_split(self, X, y, feat_idxs):
    best_gain = -1
    split_idx, split_thresh = None, None
    parent_gini = self._gini(y)

    for feat_idx in feat_idxs:
      X_column = X[:, feat_idx]
      thresholds = np.unique(X_column)

      for thresh in thresholds:
        left_mask = X_column <= thresh
        right_mask = ~left_mask

        if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
          continue

        y_left, y_right = y[left_mask], y[right_mask]
        n = len(y)
        n_l, n_r = len(y_left), len(y_right)
        child_gini = (n_l / n) * self._gini(y_left) + (n_r / n) * self._gini(
            y_right
        )
        gain = parent_gini - child_gini

        if gain > best_gain:
          best_gain = gain
          split_idx = feat_idx
          split_thresh = thresh

    return split_idx, split_thresh

  def _grow_tree(self, X, y, depth=0):
    n_samples, n_features = X.shape
    n_labels = len(np.unique(y))

    # Base stopping conditions
    if (
        depth >= self.max_depth
        or n_labels == 1
        or n_samples < self.min_samples_split
    ):
      leaf_value = np.bincount(y).argmax()
      return DecisionTreeNode(value=leaf_value)

    # 1. Feature Subsampling (Random m features)
    n_sub_features = (
        self.max_features
        if self.max_features
        else int(np.sqrt(n_features))
    )
    feat_idxs = np.random.choice(n_features, n_sub_features, replace=False)

    # 2. Greedy split search
    feat_idx, thresh = self._best_split(X, y, feat_idxs)

    if feat_idx is None:
      leaf_value = np.bincount(y).argmax()
      return DecisionTreeNode(value=leaf_value)

    # 3. Recursive subtree construction
    left_mask = X[:, feat_idx] <= thresh
    left_child = self._grow_tree(X[left_mask], y[left_mask], depth + 1)
    right_child = self._grow_tree(X[~left_mask], y[~left_mask], depth + 1)

    return DecisionTreeNode(
        feature=feat_idx,
        threshold=thresh,
        left=left_child,
        right=right_child,
    )

  def fit(self, X, y):
    self.root = self._grow_tree(X, y)

  def _predict_row(self, x, node):
    if node.is_leaf_node():
      return node.value
    if x[node.feature] <= node.threshold:
      return self._predict_row(x, node.left)
    return self._predict_row(x, node.right)

  def predict(self, X):
    return np.array([self._predict_row(x, self.root) for x in X])


class RandomForestClassifierScratch:

  def __init__(
      self,
      n_estimators=10,
      max_depth=10,
      min_samples_split=2,
      max_features=None,
  ):
    self.n_estimators = n_estimators
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.max_features = max_features
    self.trees = []

  def _bootstrap_sample(self, X, y):
    """Samples N rows with replacement."""
    n_samples = X.shape[0]
    idxs = np.random.choice(n_samples, n_samples, replace=True)
    return X[idxs], y[idxs]

  def fit(self, X, y):
    self.trees = []
    for _ in range(self.n_estimators):
      tree = DecisionTreeScratch(
          max_depth=self.max_depth,
          min_samples_split=self.min_samples_split,
          max_features=self.max_features,
      )
      # Bagging: fit each tree on a unique bootstrap sample
      X_sample, y_sample = self._bootstrap_sample(X, y)
      tree.fit(X_sample, y_sample)
      self.trees.append(tree)

  def predict(self, X):
    # Collect predictions from every tree in the forest: shape (n_trees, n_samples)
    tree_preds = np.array([tree.predict(X) for tree in self.trees])
    # Transpose to (n_samples, n_trees)
    tree_preds = np.swapaxes(tree_preds, 0, 1)
    # Majority vote for each row
    y_pred = [np.bincount(tree_votes).argmax() for tree_votes in tree_preds]
    return np.array(y_pred)


# =========================================================================
# DEMO EXECUTION
# =========================================================================
if __name__ == "__main__":
  np.random.seed(42)

  # Synthetic loan approval dataset: [Age, Income_k$, Credit_Score, Loan_Amount_k$]
  X_train = np.array([
      [25, 45, 680, 15],
      [45, 85, 750, 40],
      [35, 60, 620, 25],
      [52, 110, 800, 50],
      [23, 30, 580, 10],
      [40, 70, 710, 30],
      [60, 95, 740, 35],
      [28, 50, 640, 20],
  ])
  y_train = np.array([0, 1, 0, 1, 0, 1, 1, 0])  # 1 = Approved, 0 = Denied

  # Instantiate Random Forest with 5 trees
  rf = RandomForestClassifierScratch(n_estimators=5, max_depth=3)
  rf.fit(X_train, y_train)

  # Test applicant
  X_test = np.array([[38, 75, 720, 28]])  # Should predict 1 (Approved)
  prediction = rf.predict(X_test)

  print("=" * 60)
  print("  RANDOM FOREST ENSEMBLE CLASSIFIER (NUMPY FROM SCRATCH)")
  print("=" * 60)
  print(f"Number of Trees in Ensemble : {rf.n_estimators}")
  print(f"Query Sample Features       : {X_test[0]}")
  print(
      f"Final Ensemble Prediction   : {prediction[0]} ("
      + ("Approved" if prediction[0] == 1 else "Denied")
      + ")"
  )
  print("=" * 60)

  RANDOM FOREST ENSEMBLE CLASSIFIER (NUMPY FROM SCRATCH)
Number of Trees in Ensemble : 5
Query Sample Features       : [ 38  75 720  28]
Final Ensemble Prediction   : 1 (Approved)
